# Multi-Store, Multi-Item Sales Forecasting with XGBoost and LightGBM

This notebook provides a full pipeline for daily, weekly, and monthly sales forecasting across multiple stores and items using XGBoost and LightGBM. It covers feature engineering, model training, forecasting, aggregation, evaluation, visualization, and saving outputs.

Differences compared with modeling_9_XGBoost_daily: Re-run Optuna optimisation



## 1. Import Libraries and Load Data

Import all required libraries and load the processed dataset from a pickle file.

In [20]:
import pandas as pd
import numpy as np
import pickle
import xgboost as xgb
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import TimeSeriesSplit
import matplotlib.dates as mdates
import optuna
import random
import os
from tqdm import tqdm
import io
import joblib

# Load baseline_df
baseline_df = joblib.load('../../data/baseline.pkl')
baseline_df = baseline_df['baseline_df']
baseline_df.rename(columns={'predicted_sales': 'baseline_sales'}, inplace=True)

baseline_df['date'] = pd.to_datetime(baseline_df['date'])
baseline_df = baseline_df.sort_values(['store_nbr', 'item_nbr', 'date'])

# Load df for modeling
with open('../../data/df_processed_2.pkl', 'rb') as f: 
    df_scoped_processed = pickle.load(f)
df = df_scoped_processed['df_processed']

df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['store_nbr', 'item_nbr', 'date'])

# Ensure consistent data types for merging
df['store_nbr'] = df['store_nbr'].astype('uint8')
baseline_df['store_nbr'] = baseline_df['store_nbr'].astype('uint8')

df['item_nbr'] = df['item_nbr'].astype(str)
baseline_df['item_nbr'] = baseline_df['item_nbr'].astype(str)

df['date'] = pd.to_datetime(df['date'])
baseline_df['date'] = pd.to_datetime(baseline_df['date'])

# Merge baseline_df and df for modeling
df = df.merge(baseline_df[['store_nbr', 'item_nbr', 'date', 'baseline_sales']],
              on=['store_nbr', 'item_nbr', 'date'],
              how='left')

print(f"the min date in df is {df['date'].min()}")
print(f"the max date in df is {df['date'].max()}")


# -----------------------------
# Fix randomness for reproducibility
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# -----------------------------
# Add path to 'package' folder
# -----------------------------

import sys
import os

# Get the parent folder of the notebook
notebook_dir = os.getcwd()               # should be .../3_modeling
parent_dir = os.path.dirname(notebook_dir)  # go up one level

# Add the sibling folder 'package' to sys.path
sys.path.append(os.path.join(parent_dir, 'package'))

# Now you can import the module
from m_baseline_model import f_baseline_model
from m_metrics import f_asym_wmae, f_compute_metrics
from m_train_test_cutoff_2 import f_split_time_series_2
from m_metrics_daily import f_metrics_daily




the min date in df is 2014-01-02 00:00:00
the max date in df is 2017-08-15 00:00:00


### 1.1 Doublecheck df

In [21]:
print(df.columns)
print(df.info())         # Overview of columns, types, non-nulls
print(df.describe())     # Summary statistics for numeric columns
print(df.head())         # First 5 rows


Index(['store_nbr', 'item_nbr', 'month', 'family', 'city', 'state',
       'type_store', 'type_holiday', 'locale', 'locale_name', 'description',
       'transferred', 'unit_sales', 'id', 'onpromotion', 'day', 'class',
       'perishable', 'cluster', 'type_priority', 'temperature_2m_max',
       'salary_payment', 'magnitude', 'date', 'oil_price', 'baseline_sales'],
      dtype='object')
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1977712 entries, 0 to 1977711
Data columns (total 26 columns):
 #   Column              Dtype         
---  ------              -----         
 0   store_nbr           uint8         
 1   item_nbr            object        
 2   month               category      
 3   family              category      
 4   city                category      
 5   state               category      
 6   type_store          category      
 7   type_holiday        category      
 8   locale              category      
 9   locale_name         category      
 10  description  

## 2. Feature Engineering: Lag, Rolling, Earthquake, Oil Price and Baseline Features

Create lag features, rolling mean features, and a baseline feature for each store/item using groupby and shift/rolling operations. Define FEATURES and TARGET variables.

In [22]:

#Earthquake feature engineering
df['earthquake_event'] = (df['magnitude'] > 0).astype(int)
df['days_since_earthquake'] = df['date'] - df.loc[df['earthquake_event'] == 1, 'date'].max()
df['days_since_earthquake'] = df['days_since_earthquake'].dt.days
df['days_since_earthquake'] = df['days_since_earthquake'].where(df['earthquake_event'] == 0, 0)
df['days_since_earthquake'] = df['days_since_earthquake'].clip(lower=0)


# Ensure full daily coverage for each store-item pair
date_range = pd.date_range(df['date'].min(), df['date'].max(), freq="D")


# Build full index of all store-item-date combinations
full_index = pd.MultiIndex.from_product(
    [df['store_nbr'].unique(), df['item_nbr'].unique(), date_range],
    names=['store_nbr','item_nbr','date'])


# Aggregate once per store-item-date (ensures uniqueness)
df = (df.groupby(['store_nbr', 'item_nbr', 'date'], as_index=True)
        .agg({'unit_sales':'sum',
              'earthquake_event':'max',
              'days_since_earthquake':'max',
              'type_holiday':'first',
              'oil_price':'mean',
              'onpromotion':'last',  # or 'sum' depending on your logic
              'city':'first',
              'state':'first',
              'cluster':'first',
              'class':'first',
              'temperature_2m_max': 'max',
              'magnitude':'max',
              'salary_payment':'max',
                'baseline_sales':'first'  # from baseline model
        })
        .reindex(full_index)       # <-- reindex on full 3D index
        .reset_index())            # back to normal columns

##### PAYMENT DAY FEATURES #####
# Ensure dataframe is sorted by date
df = df.sort_values('date')

# Get all salary payment dates
salary_dates = df.loc[df['salary_payment'] == 1, 'date'].unique()

# --- 1️⃣ Compute signed distance to nearest salary date ---
def nearest_salary_distance(date, salary_dates):
    """Returns signed days between date and nearest salary date."""
    if len(salary_dates) == 0:
        return np.nan
    diffs = [(date - d).days for d in salary_dates]
    return min(diffs, key=lambda x: abs(x))

df['days_from_salary'] = df['date'].apply(lambda x: nearest_salary_distance(x, salary_dates))

# --- 2️⃣ Categorical indicators ---
# 1–3 days before payday
df['before_salary'] = df['days_from_salary'].between(-3, -1).astype(int)

# 1–3 days after payday
df['after_salary'] = df['days_from_salary'].between(0, 3).astype(int)

# --- 3️⃣ Continuous proximity effect ---
# Exponential decay to model influence of payday proximity
df['salary_proximity'] = np.exp(-0.3 * np.abs(df['days_from_salary']))

# --- 4️⃣ Optional: cyclic encoding of monthly salary patterns ---
# Helps model the repeating monthly payday cycle
df['salary_cycle_day'] = df['days_from_salary'] % 30
df['salary_cycle_sin'] = np.sin(2 * np.pi * df['salary_cycle_day'] / 30)
df['salary_cycle_cos'] = np.cos(2 * np.pi * df['salary_cycle_day'] / 30)

print("✅ Salary payment features added:",
      ['salary_payment', 'before_salary', 'after_salary',
       'salary_proximity', 'salary_cycle_sin', 'salary_cycle_cos'])


# Convert categorical columns (if using XGBoost ≥ 1.6)
for col in ['city', 'state', 'cluster', 'class', 'type_holiday']:
    df[col] = df[col].astype('category')


df['salary_payment'] = df['salary_payment'].fillna(0).astype(int)

# Now safely re-add date-based features
df['day_of_week'] = df['date'].dt.dayofweek
df['month'] = df['date'].dt.month
df['year'] = df['date'].dt.year
df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
df['days_since_start'] = (df['date'] - df['date'].min()).dt.days
df['week_of_year'] = df['date'].dt.isocalendar().week.astype(int)
df['quarter'] = df['date'].dt.quarter
df['store_mean_sales'] = df.groupby('store_nbr')['unit_sales'].transform('mean')
df['item_mean_sales'] = df.groupby('item_nbr')['unit_sales'].transform('mean')
df['store_item_ratio'] = df['store_mean_sales'] / df['item_mean_sales']


# Recreate holiday flag AFTER reindex
df['is_holiday'] = (df.get('type_holiday', "NO_HOLIDAY") != "NO_HOLIDAY").astype(int)

# Fill missing values
df['unit_sales'] = df['unit_sales'].fillna(0)   # assume no sales if missing
df['earthquake_event'] = df['earthquake_event'].fillna(0)
df['days_since_earthquake'] = df['days_since_earthquake'].fillna(method='ffill')
df['oil_price'] = df['oil_price'].fillna(method='ffill')
df['city'] = df['city'].fillna(method='ffill')
df['state'] = df['state'].fillna(method='ffill')
df['cluster'] = df['cluster'].fillna(method='ffill')
df['class'] = df['class'].fillna(method='ffill')
df['onpromotion'] = df['onpromotion'].fillna(method='ffill')
df['type_holiday'] = df['type_holiday'].fillna("NO_HOLIDAY")
df['is_holiday'] = df['is_holiday'].fillna(0)
df['temperature_2m_max'] = df['temperature_2m_max'].fillna(method='ffill')
df['magnitude'] = df['magnitude'].fillna(0)
df['salary_payment'] = df['salary_payment'].fillna(0)
df['baseline_sales'] = df['baseline_sales'].fillna(0)  # assuming 0 if missing


#Lag features and rolling means
LAGS = [7,14,21, 28, 56]
ROLLS = [7,14]

for lag in LAGS:
    df[f'lag_{lag}'] = df.groupby(['store_nbr','item_nbr'])['unit_sales'].shift(lag)
for roll in ROLLS:
    df[f'roll_mean_{roll}'] = df.groupby(['store_nbr','item_nbr'])['unit_sales'].shift(1).rolling(roll).mean()

# Roll_mean_28_lag28 feature: average sales from 29–56 days ago (relative to the current day)
df['roll_mean_28_lag28'] = df.groupby(['store_nbr','item_nbr'])['unit_sales'].shift(28).rolling(28).mean()

# Define features and target
SALARY_FEATURES = ['salary_payment', 'before_salary', 'after_salary', 'salary_proximity']
EXTRA_FEATURES = ['days_since_earthquake', 'oil_price', 'day_of_week', 'year', 'is_weekend', 'is_holiday']  + SALARY_FEATURES # add more if needed
EXTRA_FEATURES += ['days_since_start', 'week_of_year',  'month', 'quarter','store_mean_sales', 'item_mean_sales', 'store_item_ratio', 'baseline_sales']
EXTRA_FEATURES += ['city', 'state', 'cluster', 'class', 'type_holiday']  # categorical features
FEATURES = [f'lag_{lag}' for lag in LAGS] + [f'roll_mean_{roll}' for roll in ROLLS] + ['roll_mean_28_lag28'] + EXTRA_FEATURES
TARGET = 'unit_sales'

print(f'number of NaNs in features: {df[FEATURES].isna().sum().sum()}')


df_model = df


nan_counts = df_model.isna().sum()
nan_counts = nan_counts[nan_counts > 0]

print(f' NaNs broken down per feature: {nan_counts}')



nan_counts_per_group = df.groupby(['store_nbr','item_nbr'])[FEATURES].apply(lambda g: g.isna().sum())
print(nan_counts_per_group.head())





✅ Salary payment features added: ['salary_payment', 'before_salary', 'after_salary', 'salary_proximity', 'salary_cycle_sin', 'salary_cycle_cos']


C:\Users\jasmi\AppData\Local\Temp\ipykernel_8296\1497427385.py:104: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['days_since_earthquake'] = df['days_since_earthquake'].fillna(method='ffill')
C:\Users\jasmi\AppData\Local\Temp\ipykernel_8296\1497427385.py:105: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['oil_price'] = df['oil_price'].fillna(method='ffill')
C:\Users\jasmi\AppData\Local\Temp\ipykernel_8296\1497427385.py:106: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df['city'] = df['city'].fillna(method='ffill')
C:\Users\jasmi\AppData\Local\Temp\ipykernel_8296\1497427385.py:107: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[

number of NaNs in features: 237166
 NaNs broken down per feature: onpromotion           135281
lag_7                  10640
lag_14                 21280
lag_21                 31920
lag_28                 42560
lag_56                 85120
roll_mean_7             1526
roll_mean_14            1533
roll_mean_28_lag28     42587
dtype: int64
                     lag_7  lag_14  lag_21  lag_28  lag_56  roll_mean_7  \
store_nbr item_nbr                                                        
44        1014864.0      7      14      21      28      56            1   
          1014865.0      7      14      21      28      56            1   
          1018878.0      7      14      21      28      56            2   
          1044592.0      7      14      21      28      56            1   
          1052022.0      7      14      21      28      56            1   

                     roll_mean_14  roll_mean_28_lag28  days_since_earthquake  \
store_nbr item_nbr                                    

## 3. Define final validation set and training set for cross validation

In [23]:
from m_train_test_cutoff_2 import f_split_time_series_2

# Use helper function from module
df_trainval, df_test_final, cutoff_date = f_split_time_series_2(df_model, date_col="date", test_days=146, buffer_days=15)
print(f"the column names in df_model are {df_model.columns}")
print(f"the column names in df_trainval are {df_trainval.columns}")
print(f'the df_test_final runs from {df_test_final["date"].min()} to {df_test_final["date"].max()}')
print(f'the df_trainval runs from {df_trainval["date"].min()} to {df_trainval["date"].max()}')

print(f"the number of unique days in df_test_final is {df_test_final['date'].nunique()}")
print(f"the number of expected unique days in df_test_final is {(df_test_final['date'].max()-df_test_final['date'].min()).days + 1}")
print(f"the number of unique (item_nbr, store_nbr, date) combinations in df_test_final is {df_test_final[['store_nbr','item_nbr', 'date']].drop_duplicates().shape[0]}")
print(f"the number of expected (item_nbr, store_nbr, date) combinations in df_test_final is {len(df_test_final['store_nbr'].unique()) *len(df_test_final['item_nbr'].unique()) * ((df_test_final['date'].max()-df_test_final['date'].min()).days + 1)}")



the column names in df_model are Index(['store_nbr', 'item_nbr', 'date', 'unit_sales', 'earthquake_event',
       'days_since_earthquake', 'type_holiday', 'oil_price', 'onpromotion',
       'city', 'state', 'cluster', 'class', 'temperature_2m_max', 'magnitude',
       'salary_payment', 'baseline_sales', 'days_from_salary', 'before_salary',
       'after_salary', 'salary_proximity', 'salary_cycle_day',
       'salary_cycle_sin', 'salary_cycle_cos', 'day_of_week', 'month', 'year',
       'is_weekend', 'days_since_start', 'week_of_year', 'quarter',
       'store_mean_sales', 'item_mean_sales', 'store_item_ratio', 'is_holiday',
       'lag_7', 'lag_14', 'lag_21', 'lag_28', 'lag_56', 'roll_mean_7',
       'roll_mean_14', 'roll_mean_28_lag28'],
      dtype='object')
the column names in df_trainval are Index(['store_nbr', 'item_nbr', 'unit_sales', 'earthquake_event',
       'days_since_earthquake', 'type_holiday', 'oil_price', 'onpromotion',
       'city', 'state', 'cluster', 'class', 'temper

## 4. Hyperparameter Optimization with Cross-Validation (Disabled because of 45 min run time: Using Previously Saved Parameters)

In [24]:
RUN_OPTUNA = False  # <-- set to True if you want to re-run tuning
# Ensure binary/int dtype for salary_payment before tuning
df_trainval['salary_payment'] = df_trainval['salary_payment'].astype(int)

# Quick smoke-test toggles (set SMOKE_TEST=False to run full tuning)
SMOKE_TEST = True
if SMOKE_TEST:
    n_splits = 3
    test_size = 60
    num_boost_round_trial = 1000
    early_stop = 30
    n_trials = 6
else:
    n_splits = 5
    test_size = 90
    num_boost_round_trial = 5000
    early_stop = 50
    n_trials = 20

if RUN_OPTUNA:

    # --- TimeSeriesSplit for Optuna ---
    dates = df_trainval.index.unique()
    tss = TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=7)

    # --- XGBoost Objective ---
    def objective_xgb(trial):
        params = {
            "objective": "reg:squarederror",
            "eval_metric": "rmse",
            # use trial-specific seed to make parallel runs safe
            "seed": int(SEED + trial.number),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 10),
            "subsample": trial.suggest_float("subsample", 0.6, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
            "lambda": trial.suggest_float("lambda", 1e-3, 10.0, log=True),
            "alpha": trial.suggest_float("alpha", 1e-3, 10.0, log=True),
            "verbosity": 0
        }
        scores = []
        for train_idx, val_idx in tss.split(dates):
            train_dates, val_dates = dates[train_idx], dates[val_idx]
            train = df_trainval[df_trainval.index.isin(train_dates)]
            val = df_trainval[df_trainval.index.isin(val_dates)]
            X_train, y_train = train[FEATURES], np.log1p(train[TARGET])
            X_val, y_val = val[FEATURES], np.log1p(val[TARGET])
            # enable_categorical=True so pandas 'category' dtype is accepted
            dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
            dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
            model = xgb.train(params, dtrain, num_boost_round=num_boost_round_trial, evals=[(dval, "val")],
                              early_stopping_rounds=early_stop, verbose_eval=False)
            val_pred = model.predict(dval, iteration_range=(0, model.best_iteration))
            scores.append(np.sqrt(mean_squared_error(np.expm1(y_val), np.expm1(val_pred))))
        return np.mean(scores)

    # --- Run Optuna Studies ---
    study_xgb = optuna.create_study(direction="minimize", pruner=optuna.pruners.MedianPruner())
    study_xgb.optimize(objective_xgb, n_trials=n_trials, show_progress_bar=True)

    # --- Re-run CV with best params to get average best iteration ---
    best_iters_xgb = []

    # XGBoost best iters
    for train_idx, val_idx in tss.split(dates):
        train_dates, val_dates = dates[train_idx], dates[val_idx]
        train = df_trainval[df_trainval.index.isin(train_dates)]
        val = df_trainval[df_trainval.index.isin(val_dates)]
        X_train, y_train = train[FEATURES], np.log1p(train[TARGET])
        X_val, y_val = val[FEATURES], np.log1p(val[TARGET])
        dtrain = xgb.DMatrix(X_train, label=y_train, enable_categorical=True)
        dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
        model = xgb.train(study_xgb.best_params, dtrain, num_boost_round=num_boost_round_trial,
                          evals=[(dval, "val")],
                          early_stopping_rounds=early_stop, verbose_eval=False)
        best_iters_xgb.append(model.best_iteration)

    avg_best_iter_xgb_146 = int(np.mean(best_iters_xgb))
    avg_best_iter_xgb = avg_best_iter_xgb_146

    print("Average best iteration XGB:", avg_best_iter_xgb_146)

    # Example: save best params after tuning
    best_params_xgb_146 = study_xgb.best_params

     # Save to JSON
    import json
    with open("best_params_146.json", "w") as f:
        json.dump(
            {
                "xgb_146": best_params_xgb_146,
                "avg_best_iter_xgb_146": avg_best_iter_xgb_146,
            },
            f,
        )

else:
    import json
    with open("best_params_146.json", "r") as f:
        best_params = json.load(f)

    best_params_xgb_146 = best_params["xgb_146"]

    avg_best_iter_xgb = best_params.get("avg_best_iter_xgb_146", 1000)
    avg_best_iter_xgb_146 = avg_best_iter_xgb

    print("Loaded XGB params_146:", best_params_xgb_146)
    print("Using avg_best_iter_xgb_146:", avg_best_iter_xgb_146)


Loaded XGB params_146: {'learning_rate': 0.012799579682569643, 'max_depth': 7, 'subsample': 0.67336884370931, 'colsample_bytree': 0.8188334288080773, 'lambda': 0.0016522477438127444, 'alpha': 0.526251317785058}
Using avg_best_iter_xgb_146: 840


## 5. Plot showing training/val set together with final test set

In [25]:
run_this = False

if run_this: 

    # Plot setup
    tss = TimeSeriesSplit(n_splits=5, test_size=90, gap=7)
    dates = df_trainval.index.unique()
    fig, axs = plt.subplots(tss.n_splits, 1, figsize=(15, 12), sharex=True)

    for fold, (train_idx, val_idx) in enumerate(tss.split(dates)):
        train_dates = dates[train_idx]
        val_dates   = dates[val_idx]

        train = df_trainval[df_trainval.index.isin(train_dates)]
        val   = df_trainval[df_trainval.index.isin(val_dates)]

        axs[fold].plot(train.index, train['unit_sales'], label='Train', color='blue')
        axs[fold].plot(val.index, val['unit_sales'], label='Validation', color='orange')

        # Vertical line at validation start
        axs[fold].axvline(val.index.min(), color='black', linestyle='--')
        axs[fold].set_title(f'Fold {fold+1}: Train/Validation Split')
        axs[fold].legend()

    # Plot the final test set on the last fold for context
    axs[-1].plot(df_test_final.index, df_test_final['unit_sales'], 
                label='Final Test Set', color='green', linestyle='--')
    axs[-1].legend()

    # Format x-axis nicely
    for ax in axs:
        ax.xaxis.set_major_locator(mdates.MonthLocator())
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))


    for ax in axs:
        # Choose tick frequency (e.g., one tick per month)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))  # every 2 months
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.tick_params(axis='x', rotation=45)  # rotate labels for readability


    plt.tight_layout()
    plt.show()

## 6. Train on full trainval and validate on final test data

In [26]:
# ---------------------------
# Step 1: Convert relevant columns to int
# ---------------------------
df_trainval['salary_payment'] = df_trainval['salary_payment'].astype(int)
df_test_final['salary_payment'] = df_test_final['salary_payment'].astype(int)

bool_cols = ['before_salary','after_salary','is_weekend','is_holiday']
for col in bool_cols:
    df_trainval[col] = df_trainval[col].astype(int)
    df_test_final[col] = df_test_final[col].astype(int)

# ---------------------------
# Step 2: Final XGBoost parameters
# ---------------------------
params_xgb_final = best_params_xgb_146.copy()
params_xgb_final.update({
    "objective": "reg:squarederror",
    "eval_metric": "rmse",
    "seed": SEED
})

# ---------------------------
# Step 3: Prepare final datasets
# ---------------------------
X_trainval, y_trainval = df_trainval[FEATURES], np.log1p(df_trainval[TARGET])
X_test, y_test = df_test_final[FEATURES], np.log1p(df_test_final[TARGET])

# ---------------------------
# Step 4: Train Final XGBoost Model
# ---------------------------
xgb_trainval = xgb.DMatrix(X_trainval, label=y_trainval, enable_categorical=True)
final_model_xgb = xgb.train(params_xgb_final, xgb_trainval, num_boost_round=avg_best_iter_xgb, verbose_eval=False)

# ---------------------------
# Step 5: Predict on Test Set and store in df_test_final
# ---------------------------
dtest = xgb.DMatrix(X_test, enable_categorical=True)
df_test_final['pred_xgb'] = np.expm1(final_model_xgb.predict(dtest))

# ---------------------------
# Step 6: Compute final metrics
# ---------------------------
rmse_final_xgb = np.sqrt(mean_squared_error(np.expm1(y_test), df_test_final['pred_xgb']))
mae_final_xgb = mean_absolute_error(np.expm1(y_test), df_test_final['pred_xgb'])
rmse_final_baseline = np.sqrt(mean_squared_error(df_test_final['unit_sales'], df_test_final['baseline_sales']))
mae_final_baseline = mean_absolute_error(df_test_final['unit_sales'], df_test_final['baseline_sales'])

print("Final Test XGB RMSE:", rmse_final_xgb)
print("Final Test XGB MAE:", mae_final_xgb)
print("Baseline final test RMSE:", rmse_final_baseline)
print("Baseline final test MAE:", mae_final_baseline)


Final Test XGB RMSE: 7.3194489677104775
Final Test XGB MAE: 3.9505536556243896
Baseline final test RMSE: 7.560966651515021
Baseline final test MAE: 4.258852481842041


## 7. Create  new Baseline dynamic function for calculating baselines for 90 days in the future

In [27]:
def f_rolling_baseline_dynamic(history_df: pd.DataFrame, future_df: pd.DataFrame, 
                               lookback_days: int = 28, n_last_same_weekdays: int = 4):
    """
    Compute rolling baseline recursively for future dates.
    Avoids zero collapse by using prior baseline predictions as history.
    """
    history_df = history_df.copy()
    future_df = future_df.copy()
    
    # Ensure correct types
    history_df['date'] = pd.to_datetime(history_df['date'])
    future_df['date'] = pd.to_datetime(future_df['date'])
    for col in ['store_nbr', 'item_nbr']:
        history_df[col] = history_df[col].astype(str)
        future_df[col] = future_df[col].astype(str)
    
    # Initialize list to store results
    baseline_rows = []

    # Group history by store/item for fast access
    history_grouped = history_df.groupby(['store_nbr','item_nbr'])

    # Iterate over each store/item
    for store, item in tqdm(future_df[['store_nbr','item_nbr']].drop_duplicates().values,
                            desc="Baseline forecast"):
        # Get past history
        if (store, item) in history_grouped.groups:
            hist_sales = history_grouped.get_group((store, item)).sort_values('date')['unit_sales'].tolist()
            hist_dates = history_grouped.get_group((store, item)).sort_values('date')['date'].tolist()
        else:
            hist_sales = []
            hist_dates = []
        
        # Track history including predicted baseline
        history_sales = hist_sales.copy()
        history_dates = hist_dates.copy()

        # Filter future dates for this store/item
        future_dates_item = future_df[(future_df['store_nbr']==store) & (future_df['item_nbr']==item)].sort_values('date')['date']

        # Iterate recursively through future dates
        for future_date in future_dates_item:
            weekday = future_date.weekday()  # 0=Mon,6=Sun

            # Last n_last_same_weekdays values for this weekday
            past_same_weekday_sales = [
                s for d, s in zip(history_dates, history_sales) if d.weekday() == weekday
            ]
            last_sales = past_same_weekday_sales[-n_last_same_weekdays:] if len(past_same_weekday_sales) >= 1 else [0.0]

            # Compute baseline as mean of last same-weekday sales
            baseline = float(np.mean(last_sales))

            # Store result
            baseline_rows.append({
                'store_nbr': store,
                'item_nbr': item,
                'date': future_date,
                'baseline_sales': baseline
            })

            # Append baseline to history for next iteration
            history_sales.append(baseline)
            history_dates.append(future_date)

    # Build final DataFrame
    baseline_df = pd.DataFrame(baseline_rows)
    return baseline_df.rename(columns={'baseline_sales':'pred_baseline'})




## 8. Daily 90-Day Forecast per Store/Item - XGBoost including baseline

For each store/item, generate a 90-day daily forecast using the trained models, updating lag/rolling features iteratively.

In [28]:
RUN_90_days_forecast = False  # <-- set to True if you want to re-run 

if RUN_90_days_forecast :


    # --- 0. Ensure consistent key types (critical fix) ---
    df_model['store_nbr'] = df_model['store_nbr'].astype(str).str.strip()
    df_model['item_nbr'] = (
        df_model['item_nbr']
        .astype(str)
        .str.strip()
        .str.replace(r'\.0$', '', regex=True)
    )

    # --- 1. Ensure date column ---
    if 'date' not in df_model.columns:
        df_model = df_model.reset_index()
    df_model['date'] = pd.to_datetime(df_model['date'])

    # --- 2. Define LAGS & ROLLS if not already defined ---
    try:
        LAGS
    except NameError:
        LAGS = [7, 14, 21, 28]
    try:
        ROLLS
    except NameError:
        ROLLS = [7, 14, 28]

    # --- 3. Future horizon setup ---
    future_horizon = 90
    last_date = df_model['date'].max()

    # Create store-item pairs AFTER cleaning
    stores_items = df_model[['store_nbr', 'item_nbr']].drop_duplicates()

    # Create future dates & cross-merge
    future_dates = pd.date_range(start=last_date + pd.Timedelta(days=1), periods=future_horizon)
    future_dates_df = pd.DataFrame({'date': future_dates})
    future_df = stores_items.merge(future_dates_df, how='cross')
    future_df['unit_sales'] = 0

    # Clean up again to be extra safe
    for col in ['store_nbr', 'item_nbr']:
        future_df[col] = (
            future_df[col]
            .astype(str)
            .str.strip()
            .str.replace(r'\.0$', '', regex=True)
        )

    # --- 4. Compute recursive dynamic baseline ---
    baseline_preds = f_rolling_baseline_dynamic(df_model, future_df)

    # --- 5. Build history dictionary ---
    df_model_sorted = df_model.sort_values(['store_nbr', 'item_nbr', 'date'])
    history_dict = {
        (store, item): v['unit_sales'].values.astype(float)
        for (store, item), v in df_model_sorted.groupby(['store_nbr', 'item_nbr'])
    }
    min_history_len = max(max(LAGS), max(ROLLS), 28)

    # --- 6. Precompute static + date features ---
    last_attrs = df_model.groupby(['store_nbr', 'item_nbr']).last().reset_index()
    future_static = future_df.merge(
        last_attrs[['store_nbr', 'item_nbr']],
        on=['store_nbr', 'item_nbr'], how='left'
    )
    future_static['day_of_week'] = future_static['date'].dt.dayofweek
    future_static['month'] = future_static['date'].dt.month
    future_static['year'] = future_static['date'].dt.year
    future_static['is_weekend'] = future_static['day_of_week'].isin([5,6]).astype(int)
    future_static['days_since_earthquake'] = 0.0
    future_static['oil_price'] = 0.0
    future_static['is_holiday'] = 0

    future_static_dict_local = {
        (row['store_nbr'], row['item_nbr'], row['date']): {
            'day_of_week': row['day_of_week'],
            'month': row['month'],
            'year': row['year'],
            'is_weekend': row['is_weekend'],
            'days_since_earthquake': row['days_since_earthquake'],
            'oil_price': row['oil_price'],
            'is_holiday': row['is_holiday'],
        }
        for _, row in future_static.iterrows()
    }

    # --- 7. Define feature list ---
    FEATURES_local = (
        [f'lag_{lag}' for lag in LAGS]
        + [f'roll_mean_{roll}' for roll in ROLLS]
        + ['roll_mean_28_lag28']
        + [
            'days_since_earthquake',
            'oil_price',
            'day_of_week',
            'month',
            'year',
            'is_weekend',
            'is_holiday',
        ]
    )

    # --- 8. Recursive forecasting with XGB ---
    rows = []
    for store, item in tqdm(stores_items.values, desc="Forecasting store-items"):
        key = (store, item)
        history_arr = history_dict.get(key)
        if history_arr is None or len(history_arr) < min_history_len:
            continue

        history = history_arr.tolist()

        for future_date in future_dates:
            # 1. Compute lags
            X_pred = {f'lag_{lag}': history[-lag] for lag in LAGS}

            # 2. Rolling means
            for roll in ROLLS:
                X_pred[f'roll_mean_{roll}'] = float(np.mean(history[-roll:]))

            # 3. 28-day back window mean
            X_pred['roll_mean_28_lag28'] = (
                float(np.mean(history[-56:-28])) if len(history) >= 56 else 0.0
            )

            # 4. Static/date features
            feats = future_static_dict_local.get((store, item, future_date), {})
            X_pred.update(feats)

            # 5. Predict
            df_row = pd.DataFrame([{feat: X_pred.get(feat, 0.0) for feat in FEATURES_local}])
            dmatrix = xgb.DMatrix(df_row)
            pred_raw = final_model_xgb.predict(dmatrix)[0]
            pred_xgb = float(np.expm1(pred_raw))

            # 6. Store
            rows.append({
                'store_nbr': store,
                'item_nbr': item,
                'date': future_date,
                'pred_xgb': pred_xgb
            })

            # 7. Recursive update
            history.append(pred_xgb)

    # --- 9. Merge baseline ---
    if len(rows) == 0:
        print("⚠️ No forecasts were generated (check min_history_len and data availability).")
        daily_forecast_xgb = pd.DataFrame(columns=['store_nbr','item_nbr','date','pred_xgb','pred_baseline'])
    else:
        daily_forecast_xgb = pd.DataFrame(rows)
        daily_forecast_xgb['date'] = pd.to_datetime(daily_forecast_xgb['date'])
        daily_forecast_xgb = daily_forecast_xgb.merge(
            baseline_preds[['store_nbr','item_nbr','date','pred_baseline']],
            on=['store_nbr','item_nbr','date'], how='left'
        )

        display(daily_forecast_xgb.head(20))
        print(daily_forecast_xgb.columns)
        print(daily_forecast_xgb.dtypes)


### Calculate performance metrics for the daily predictions of next week

##### Calculate performance metrics for the daily predictions of the next ISO-week at store-item level. This performance evaluation is relevant for the store manager

In [29]:


metrics_combined = f_metrics_daily(
    df_test_final,
    y_true_col='unit_sales',
    y_pred_cols=['pred_xgb', 'baseline_sales'])

metrics_rounded = metrics_combined.round({
    'rmse': 2,
    'mae': 2,
    'mape': 2,
    'rae': 2,
    'bias': 2,
    'over_delivery': 0,
    'under_delivery': 0,
    'total_sales': 0})


print(metrics_rounded)
print(metrics_rounded.to_csv(index=False))




       prediction  rmse   mae        mape        rae   bias  over_delivery  \
0  baseline_sales  6.68  3.95  48504248.0  45.369999  -9.27        24124.0   
1        pred_xgb  6.48  3.66  31982014.0  42.009998  11.86        13456.0   

   under_delivery  total_sales     week  
0         16122.0      88694.0  Average  
1         23921.0      88694.0  Average  
prediction,rmse,mae,mape,rae,bias,over_delivery,under_delivery,total_sales,week
baseline_sales,6.68,3.95,48504250.0,45.37,-9.27,24124.0,16122.0,88694.0,Average
pred_xgb,6.48,3.66,31982014.0,42.01,11.86,13456.0,23921.0,88694.0,Average



## Check for correlations between features and the target

In [30]:
# Select only numeric columns from your features
numeric_features = df_model[FEATURES].select_dtypes(include=[np.number])

# Then compute correlations
corrs = numeric_features.corrwith(df_model[TARGET], numeric_only=True).sort_values(ascending=False)

print(corrs)



lag_7                    0.785293
lag_14                   0.758934
lag_21                   0.736783
lag_28                   0.729556
lag_56                   0.713341
item_mean_sales          0.578734
roll_mean_7              0.288483
roll_mean_14             0.205342
store_mean_sales         0.204058
baseline_sales           0.202002
roll_mean_28_lag28       0.199794
is_weekend               0.158137
day_of_week              0.122890
after_salary             0.022294
salary_payment           0.019513
days_since_start         0.017392
year                     0.016510
salary_proximity         0.006132
month                    0.004864
quarter                  0.003286
week_of_year            -0.001728
before_salary           -0.014164
days_since_earthquake   -0.016130
is_holiday              -0.016196
oil_price               -0.020735
store_item_ratio        -0.268372
dtype: float64


In [31]:
# 1) How many baseline predictions are exactly zero?
print("pred_baseline zeros:", (df_test_final['pred_baseline']==0).sum(), " / ", len(df_test_final))

# 2) Are many store-item-date rows originally unobserved (created by reindex)?
print("Proportion of rows with unit_sales == 0 in df_model:", (df_model['unit_sales'] == 0).mean())

# 3) Compare baseline medians for observed vs reindexed rows (if you have an 'observed' flag)
if 'observed' in df_model.columns:
    print("median baseline observed rows:", df_model.loc[df_model['observed'], 'baseline_sales'].median())
    print("median baseline unobserved rows:", df_model.loc[~df_model['observed'], 'baseline_sales'].median())


KeyError: 'pred_baseline'

In [ ]:
# Check number of store-item pairs that ever sold anything
active_pairs = df_model.groupby(['store_nbr','item_nbr'])['unit_sales'].sum().gt(0).sum()
total_pairs = df_model.groupby(['store_nbr','item_nbr']).ngroups
print(f"Active pairs: {active_pairs} / {total_pairs}")

# Check how many history rows actually exist before forecast start
print("Latest date in df_model:", df_model['date'].max())
print("Earliest future date:", future_df['date'].min())


## 8. Plot Forecasts - BASELINE, XGB


Plot total daily, weekly, and monthly forecasts using matplotlib for visual inspection.

In [ ]:
def plot_forecast(df_forecast, date_col='date', value_col='pred_xgb', title='Forecast'):
    plt.figure(figsize=(12, 5))
    plt.plot(df_forecast[date_col], df_forecast[value_col], marker='o')
    plt.title(title)
    plt.xlabel('Date')
    plt.ylabel('Sales')
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.show()

# Plot total daily forecast
plot_forecast(daily_forecast_xgb.groupby('date')['pred_xgb'].sum().reset_index(), title='Total Daily Forecast (sum all stores/items)')

# Plot weekly forecast
plot_forecast(weekly_forecast_2weeks_xgb, date_col='week', title='Weekly Forecast (per store)')

# Plot monthly forecast
plot_forecast(monthly_forecast_2months_xgb, date_col='month', title='Monthly Forecast (per item)')

## 8.2 Plot Daily Total Sales + Forecasts - BASELINE, XGB

In [ ]:
# Plot total sales: actuals (train/test), test predictions, and forecast

train_total = df_trainval.groupby('date')['unit_sales'].sum().reset_index()
test_total = df_test_final.groupby('date')[['unit_sales', 'pred_lgb']].sum().reset_index()
forecast_total = daily_forecast_xgb.groupby('date')['pred_lgb'].sum().reset_index()

plt.figure(figsize=(14, 6))
plt.plot(train_total['date'], train_total['unit_sales'], label='Train Actual', color='blue')
plt.plot(test_total['date'], test_total['unit_sales'], label='Test Actual', color='green')
plt.plot(test_total['date'], test_total['pred_xgb'], label='Test Prediction (XGB)', color='orange')
plt.plot(forecast_total['date'], forecast_total['pred_xgb'], label='7-Day Forecast (XGB)', color='red', marker='o')

plt.title('Total Sales: Actuals, Predictions, and 7-Day Forecast')
plt.xlabel('Date')
plt.ylabel('Sales')
plt.legend()
plt.grid(True)
plt.show()

## 8.3 Plot Daily Total Sales + Forecasts for an ITEM/STORE combination - BASELINE, XGB

In [ ]:
# Plot actuals and predictions for train, test, and forecast for a single store/item
from datetime import datetime
store_id = 44
item_id = 1971328

# Get data for this store/item
df_train = df_trainval[(df_trainval['store_nbr'] == store_id) & (df_trainval['item_nbr'] == item_id)].copy()
df_test = df_test_final[(df_test_final['store_nbr'] == store_id) & (df_test_final['item_nbr'] == item_id)].copy()
df_forecast_xgb = daily_forecast_xgb[(daily_forecast_xgb['store_nbr'] == store_id) & (daily_forecast_xgb['item_nbr'] == item_id)].copy()
df_train = df_train.reset_index().copy()
df_test = df_test.reset_index().copy()
df_forecast_xgb = df_forecast_xgb.reset_index().copy()

plt.figure(figsize=(14, 6))


plt.plot(df_train['date'], df_train['unit_sales'], label='Train Actual', color='blue')
plt.plot(df_test['date'], df_test['unit_sales'], label='Test Actual', color='green')
plt.plot(df_test['date'], df_test['pred_xgb'], label='Test Prediction (XGB)', color='orange')
plt.plot(df_forecast_xgb['date'], df_forecast_xgb['pred_xgb'], label='7-Day Forecast (XGB)', color='red', marker='o')

plt.title(f'Store {store_id}, Item {item_id}: Actuals, Predictions, and Forecast')
plt.xlabel('Date')
plt.xlim(left=datetime(2017, 3, 1), right=datetime(2017,11,1) )
plt.ylabel('Sales')
plt.legend()
plt.grid(True)
plt.show()

## 9. Save Forecast Outputs

Save the daily, weekly, and monthly forecast DataFrames to CSV files for downstream use.

In [ ]:
daily_forecast_xgb.to_csv('../data/daily_forecast_xgb.csv', index=False)
weekly_forecast_2weeks_xgb.to_csv('../data/weekly_forecast_xgb.csv', index=False)
monthly_forecast_2months_xgb.to_csv('../data/monthly_forecast_xgb.csv', index=False)

